In [ ]:
import pandas as pd

# Read the first CSV file into a DataFrame
df_single_val_repairs = pd.read_csv("../../datasets/single_value_repairs.csv")

df_single_val_repairs

In [ ]:
df_single_val_repairs.iloc[0]['wds1']

In [ ]:
import requests
import xml.etree.ElementTree as ET

def statementDeleted(obj_statement):
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    # SPARQL query
    query = f"""ASK {{ <{obj_statement}> ?p ?o  }}
            """

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            #print(boolean_element.text)
            return boolean_element.text.lower() == 'false'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None

# Example usage
obj_statement = "http://www.wikidata.org/entity/statement/Q1001-6408985C-EAAD-4193-95F7-FCF2B179E6E3"
print(statementDeleted(obj_statement))

In [ ]:
df_single_val_repairs["A-box statement Deleted"] = None

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os

# Load the data, or resume from the last checkpoint
checkpoint_file = "/home/jovyan/work/repairs/single_value/checkpoint.csv"
if os.path.exists(checkpoint_file):
    df_single_val_repairs = pd.read_csv(checkpoint_file)
    print("Resuming from the last checkpoint.")
else:
    print("Starting from scratch.")

# Process the rows with progress bar
for index, row in tqdm(df_single_val_repairs.iterrows(), total=len(df_single_val_repairs)):
    
    # Check for unprocessed rows
    if pd.isna(row['A-box statement Deleted']):
        result = statementDeleted(row['wds1'])
        df_single_val_repairs.at[index, 'A-box statement Deleted'] = result

    # Save a checkpoint every 10,000 rows
    if index % 10000 == 0:
        df_single_val_repairs.to_csv(checkpoint_file, index=False)
        print(f"Checkpoint saved at row {index}.")

# Save the final output
df_single_val_repairs.to_csv(checkpoint_file, index=False)
print("Processing complete. Final output saved.")

In [ ]:
df_single_val_repairs

In [ ]:
len(df_single_val_repairs[(df_single_val_repairs['A-box statement Deleted'] == True)] )

In [ ]:
df_single_val_repairs.to_csv("single_value_all_repairs.csv",index=False)

In [ ]:
len(df_single_val_repairs[(df_single_val_repairs['Constraint Deleted'] == True)] )

In [ ]:
len(df_single_val_repairs[(df_single_val_repairs['Constraint Deprecated'] == True)] )

In [ ]:
len(df_single_val_repairs[(df_single_val_repairs['Included as Exception'] == True)] )

In [ ]:
df_single_val_repairs[(df_single_val_repairs['Constraint Deleted'] == False) & 
     (df_single_val_repairs['Constraint Deprecated'] == False)& 
     (df_single_val_repairs['Included as Exception'] == False)& 
     (df_single_val_repairs['A-box statement Deleted'] == False) 
     
    ]

In [ ]:
import requests
import xml.etree.ElementTree as ET

def complementaryStatementDeleted(subject, wd_pid):
    
    p_pid = wd_pid.replace("http://www.wikidata.org/entity/", "http://www.wikidata.org/prop/")
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    # SPARQL query
    query = f"""SELECT (COUNT( DISTINCT(?o)) AS ?total)
        WHERE
        {{
          <{subject}> <{p_pid}> ?o
        }}
    """

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)

        # Define the namespace used in the XML
        namespace = {'ns': 'http://www.w3.org/2005/sparql-results#'}

        # Find the 'literal' element containing the 'total' value
        literal_element = root.find('.//ns:literal', namespace)

        # Extract the text from the 'literal' element and convert to an integer
        total_value = int(literal_element.text) if literal_element is not None else 0

        # Return True if total is 1, False otherwise
        result = total_value == 1
        
        return result
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None

# Example usage
subject = "http://www.wikidata.org/entity/Q1048"
wd_pid = "http://www.wikidata.org/entity/P1003"
print(complementaryStatementDeleted(subject, wd_pid))


In [ ]:
type(df_single_val_repairs.iloc[0]["A-box statement Deleted"])

In [ ]:
df_single_val_repairs["A-box complementary statement Deleted"] = None

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os

# Load the data, or resume from the last checkpoint
#checkpoint_file = "/home/jovyan/work/repairs/single_value/checkpoint.csv"
#if os.path.exists(checkpoint_file):
#    df_single_val_repairs = pd.read_csv(checkpoint_file)
#    print("Resuming from the last checkpoint.")
#else:
#    print("Starting from scratch.")

# Process the rows with progress bar
for index, row in tqdm(df_single_val_repairs.iterrows(), total=len(df_single_val_repairs)):
    
    # Check for unprocessed rows
    if pd.isna(row['A-box complementary statement Deleted']):
        if row['A-box statement Deleted'] is True:
            df_single_val_repairs.at[index, 'A-box complementary statement Deleted'] = False
        else:      
            result = complementaryStatementDeleted(row['subject'], row['property'])
            df_single_val_repairs.at[index, 'A-box complementary statement Deleted'] = result

        # Save a checkpoint every 10,000 rows
        if index % 10000 == 0:
            df_single_val_repairs.to_csv(checkpoint_file, index=False)
            print(f"Checkpoint saved at row {index}.")

# Save the final output
df_single_val_repairs.to_csv(checkpoint_file, index=False)
print("Processing complete. Final output saved.")

In [ ]:
df_single_val_repairs

In [ ]:
df_single_val_repairs["A-box complementary statement Deleted"].value_counts()

In [ ]:
df_single_val_repairs[(df_single_val_repairs['A-box statement Deleted'] == False) & 
     (df_single_val_repairs['no_separator'] == False )& 
     (df_single_val_repairs['T-box separator added to constraint'] == False)#& 
     #(df_single_val_repairs['A-box complementary statement Deleted'] == False)
]

In [ ]:
df_single_val_repairs.iloc[236962]['wds1']

In [ ]:
df_single_val_repairs.iloc[236962]

In [ ]:
import requests
import xml.etree.ElementTree as ET

def getConstraintSeparator(wd_pid, endpoint = "ENTER_qEndpoint_WD_2023"):

    # SPARQL query
    query = f"""
    PREFIX wikibase: <http://wikiba.se/ontology#>
    SELECT ?separator
        WHERE
        {{
          <{wd_pid}> <http://www.wikidata.org/prop/P2302> ?constraint.
          ?constraint <http://www.wikidata.org/prop/statement/P2302> <http://www.wikidata.org/entity/Q19474404>.
          ?constraint <http://www.wikidata.org/prop/qualifier/P4155>/wikibase:qualifier ?separator.

          FILTER NOT EXISTS {{?constraint <http://www.wikidata.org/prop/qualifier/P2241> []}}
          FILTER NOT EXISTS {{?constraint wikibase:rank wikibase:DeprecatedRank}}
        }}
    """

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)

        # Define the namespace used in the XML
        namespace = {'ns': 'http://www.w3.org/2005/sparql-results#'}

        # Find all 'uri' elements and extract their text
        uris = [uri_element.text for uri_element in root.findall('.//ns:uri', namespace)]

        if len(uris) == 0:
            return []
        
        return uris
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None

def hasSeparator(triple_statement, separator, endpoint):
    # SPARQL query
    query = f"""ASK {{ <{triple_statement}> <{separator}> ?o  }}
            """

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            #print(boolean_element.text)
            return boolean_element.text.lower() == 'true'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None
    

def wasSeparatorValueAdded(wd_pid, triple_statement):
    
    # URL of the endpoint
    endpoint_19 = "ENTER_qEndpoint_WD_2019"
    endpoint_23 = "ENTER_qEndpoint_WD_2023"

    separators = getConstraintSeparator(wd_pid)
    if separators:
        for separator in separators:
            if not hasSeparator(triple_statement, separator, endpoint_19) and hasSeparator(triple_statement, separator, endpoint_23):
                  return True
    return False

   # Example usage
wd_pid = "http://www.wikidata.org/entity/P1006"
print(getConstraintSeparator(wd_pid))

stmt = "http://www.wikidata.org/entity/statement/Q12998-2619692C-2C47-4E03-A95B-518E02281017"
wasSeparatorValueAdded(wd_pid, stmt)

In [ ]:
df_single_val_repairs["A-box separator added"] = None

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os


# Process the rows with progress bar
for index, row in tqdm(df_single_val_repairs.iterrows(), total=len(df_single_val_repairs)):
    
    # Check for unprocessed rows
    if pd.isna(row["A-box separator added"]):
        if row['A-box statement Deleted'] is True:
            df_single_val_repairs.at[index, 'A-box separator added'] = False
        elif row['no_separator'] is False and row['T-box separator added to constraint'] is False:      
            result = wasSeparatorValueAdded(row['property'], row['wds1'])
            df_single_val_repairs.at[index, 'A-box separator added'] = result

        # Save a checkpoint every 10,000 rows
        if index % 10000 == 0:
            df_single_val_repairs.to_csv(checkpoint_file, index=False)
            print(f"Checkpoint saved at row {index}.")

# Save the final output
df_single_val_repairs.
(checkpoint_file, index=False)
print("Processing complete. Final output saved.")

In [ ]:
indices = df_single_val_repairs[df_single_val_repairs['property'].isna()].index

indices

In [ ]:
df_single_val_repairs["A-box separator added"].value_counts()

In [ ]:
df_single_val_repairs[(df_single_val_repairs['A-box separator added'] == True)
]

In [ ]:
df_single_val_repairs[(df_single_val_repairs['A-box separator added'] == None)
]

In [ ]:
df_single_val_repairs.iloc[238692]

In [ ]:
df_single_val_repairs.iloc[238692]['wds1']

In [ ]:
getConstraintSeparator("http://www.wikidata.org/entity/P1081")

In [ ]:
def getCountSeparatorValue(wds1, subject, wd_pid, pq_qua, endpoint):

    p_pid = wd_pid.replace("http://www.wikidata.org/entity/", "http://www.wikidata.org/prop/")
    
    query = f"""
        PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        PREFIX wikibase: <http://wikiba.se/ontology#>
        PREFIX p: <http://www.wikidata.org/prop/>
        PREFIX pq: <http://www.wikidata.org/prop/qualifier/>
        PREFIX ps: <http://www.wikidata.org/prop/statement/>
        PREFIX wd: <http://www.wikidata.org/entity/>
        PREFIX wdt: <http://www.wikidata.org/prop/direct/>

        SELECT (COUNT(?o) AS ?total)
        WHERE
        {{
            <{wds1}> <{pq_qua}> ?o.
            <{subject}> <{p_pid}>/<{pq_qua}> ?o
        }}
    """
    
    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)

        # Define the namespace used in the XML
        namespace = {'ns': 'http://www.w3.org/2005/sparql-results#'}

        # Find the 'literal' element containing the 'total' value
        literal_element = root.find('.//ns:literal', namespace)

        # Extract the text from the 'literal' element and convert to an integer
        total_value = int(literal_element.text) if literal_element is not None else 0
        
        return total_value
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None
    
    

def checkIfComplementaryWithSeparatorWasRemoved(row):
    
    pq_qua_list = getConstraintSeparator(row['property'])
    if pq_qua_list is None:
        return False
    for pq_qua in pq_qua_list:
        count_2019 = getCountSeparatorValue(row['wds1'], row['subject'], row['property'], pq_qua, "ENTER_qEndpoint_WD_2019")
        count_2023 = getCountSeparatorValue(row['wds1'], row['subject'], row['property'], pq_qua, "ENTER_qEndpoint_WD_2023")
        if count_2023 < count_2019 and count_2023 == 1:
            return True
    return False

In [ ]:
checkIfComplementaryWithSeparatorWasRemoved(df_single_val_repairs.iloc[238692])

In [ ]:
df_single_val_repairs["A-box complementary statement Deleted"].value_counts()

In [ ]:
df_single_val_repairs["A-box complementary statement Deleted"].value_counts()

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os


# Process the rows with progress bar
for index, row in tqdm(df_single_val_repairs.iterrows(), total=len(df_single_val_repairs)):
    
    # Check for unprocessed rows
    if row["A-box complementary statement Deleted"] is False:
        if row['A-box statement Deleted'] is False and row['no_separator'] is False and row['T-box separator added to constraint'] is False and row['A-box separator added'] is False :      
            result = checkIfComplementaryWithSeparatorWasRemoved(row)
            df_single_val_repairs.at[index, 'A-box complementary statement Deleted'] = result

        # Save a checkpoint every 10,000 rows
        if index % 10000 == 0:
            df_single_val_repairs.to_csv(checkpoint_file, index=False)
            print(f"Checkpoint saved at row {index}.")

# Save the final output
df_single_val_repairs.to_csv(checkpoint_file, index=False)
print("Processing complete. Final output saved.")

In [ ]:
df_single_val_repairs[
     (df_single_val_repairs['no_separator'] == False)  
    & (df_single_val_repairs['T-box separator added to constraint'] == False)#& 
     #(df_single_val_repairs['Constraint Deleted'] == False)& 
     #(df_single_val_repairs['Constraint Deprecated'] == False)
    &(df_single_val_repairs['A-box statement Deleted'] == False) & 
     (df_single_val_repairs['A-box complementary statement Deleted'] == False) #
    & (df_single_val_repairs['A-box separator added'] == False) 
     
    ]

In [ ]:
df_single_val_repairs[
     #(df_single_val_repairs['no_separator'] == False)  
    # (df_single_val_repairs['T-box separator added to constraint'] == False)
    (df_single_val_repairs['Constraint Deleted'] == False)
    & (df_single_val_repairs['Constraint Deprecated'] == False)
    & (df_single_val_repairs['Included as Exception'] == False)
    & (df_single_val_repairs['A-box statement Deleted'] == False) 
    & (df_single_val_repairs['A-box complementary statement Deleted'] == False) #
    & (df_single_val_repairs['A-box separator added'] == False) 
     
    ]

In [ ]:
df_single_val_repairs.iloc[240120]

In [ ]:
df_single_val_repairs.iloc[240120]['wds1']

In [ ]:
getConstraintSeparator("http://www.wikidata.org/entity/P1082", "ENTER_qEndpoint_WD_2019")

In [ ]:
getConstraintSeparator("http://www.wikidata.org/entity/P1082", "ENTER_qEndpoint_WD_2023")

In [ ]:
import requests
import xml.etree.ElementTree as ET

def getTripleSeparators(wds1, endpoint = "ENTER_qEndpoint_WD_2023"):

    # SPARQL query
    query = f"""
    PREFIX wikibase: <http://wikiba.se/ontology#>
    SELECT
      ?p
    WHERE
    {{
       <{wds1}> ?p ?o.
      [] wikibase:qualifier ?p
    }}
    """

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)

        # Define the namespace used in the XML
        namespace = {'ns': 'http://www.w3.org/2005/sparql-results#'}

        # Find all 'uri' elements and extract their text
        uris = [uri_element.text for uri_element in root.findall('.//ns:uri', namespace)]

        if len(uris) == 0:
            return None
        
        return uris
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None

In [ ]:
getTripleSeparators("http://www.wikidata.org/entity/statement/Q1000029-02ED7526-768E-47C1-BE19-F6BDB8110C07")

In [ ]:
df_single_val_repairs["T-box separator added"] = None

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os


# Process the rows with progress bar
for index, row in tqdm(df_single_val_repairs.iterrows(), total=len(df_single_val_repairs)):
    
    df_single_val_repairs.at[index, 'T-box separator added'] = testTboxseparatorAdded(row)

    # Save a checkpoint every 10,000 rows
    if index % 50000 == 0:
        df_single_val_repairs.to_csv(checkpoint_file, index=False)
        print(f"Checkpoint saved at row {index}.")

# Save the final output
df_single_val_repairs.to_csv(checkpoint_file, index=False)
print("Processing complete. Final output saved.")

In [ ]:
def testTboxseparatorAdded(row):
    if row["A-box statement Deleted"] is False:
        c_separators_2023 = getConstraintSeparator(row["property"], "ENTER_qEndpoint_WD_2023")
        if len(c_separators_2023) == 0:
            return False
        else:
            c_separators_2019 = getConstraintSeparator(row["property"], "ENTER_qEndpoint_WD_2019")
            if len(c_separators_2019) == 0:
                c_added_separators = c_separators_2023
            else:
                # A list of separators in 2023 who were not in 2019
                c_added_separators = [item for item in c_separators_2023 if item not in c_separators_2019]

            intance_separators = getTripleSeparators(row["wds1"])
            if intance_separators is None or len(intance_separators) == 0:
                return False
            else:
                for sep in intance_separators:
                    if sep in c_added_separators:
                        return True
    return False

In [ ]:
testTboxseparatorAdded(df_single_val_repairs.iloc[1])

In [ ]:
df_single_val_repairs

In [ ]:
df_single_val_repairs["T-box separator added"].value_counts()

In [ ]:
df_single_val_repairs["T-box separator added"].value_counts()

In [ ]:
df_single_val_repairs[
     #(df_single_val_repairs['no_separator'] == False)  
    # (df_single_val_repairs['T-box separator added to constraint'] == False)
    (df_single_val_repairs['Constraint Deleted'] == False)
    & (df_single_val_repairs['Constraint Deprecated'] == False)
    & (df_single_val_repairs['Included as Exception'] == False)
    & (df_single_val_repairs['A-box statement Deleted'] == False) 
    & (df_single_val_repairs['A-box complementary statement Deleted'] == False) #
    & (df_single_val_repairs['A-box separator added'] == False) 
    & (df_single_val_repairs['T-box separator added'] == False) 
     
    ]

In [ ]:
df_single_val_repairs.iloc[240163]

In [ ]:
# it had 1 separator and more separators were added: P459, 518

df_single_val_repairs.iloc[240163]['wds1']

In [ ]:
df_single_val_repairs.to_csv("single_value_all_repairs.csv",index=False)

In [ ]:
def getStatementsList(row, endpoint = "ENTER_qEndpoint_WD_2023"):

    p_pid = row['property'].replace("http://www.wikidata.org/entity/", "http://www.wikidata.org/prop/")
    # SPARQL query
    query = f"""
    PREFIX wikibase: <http://wikiba.se/ontology#>
    SELECT ?o
        WHERE
        {{
          <{row['subject']}> <{p_pid}> ?o.
        }}
    """

    # URL encode the query
    encoded_query = requests.utils.quote(query)

    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"

    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}

    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)

        # Define the namespace used in the XML
        namespace = {'ns': 'http://www.w3.org/2005/sparql-results#'}

        # Find all 'uri' elements and extract their text
        uris = [uri_element.text for uri_element in root.findall('.//ns:uri', namespace)]

        
        if len(uris) == 0:
            return []
        
        uris.remove(row['wds1'])
        return uris
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None
    
def checkIfComplementaryWasRemoved(row):
    statement_list = getStatementsList(row, "ENTER_qEndpoint_WD_2019")
    for stmt in statement_list:
        if statementDeleted(stmt):
            return True
    return False

checkIfComplementaryWasRemoved(df_single_val_repairs.iloc[240163])

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os


# Process the rows with progress bar
for index, row in tqdm(df_single_val_repairs.iterrows(), total=len(df_single_val_repairs)):
    
    # Check for unprocessed rows
    if row["A-box complementary statement Deleted"] is False:
        if row['A-box statement Deleted'] is False and row['no_separator'] is False and row['A-box separator added'] is False :      
            result = checkIfComplementaryWasRemoved(row)
            df_single_val_repairs.at[index, 'A-box complementary statement Deleted'] = result

        # Save a checkpoint every 10,000 rows
        if index % 10000 == 0:
            df_single_val_repairs.to_csv(checkpoint_file, index=False)
            print(f"Checkpoint saved at row {index}.")

# Save the final output
df_single_val_repairs.to_csv(checkpoint_file, index=False)
print("Processing complete. Final output saved.")

In [ ]:
df_single_val_repairs["A-box complementary statement Deleted"].value_counts()

In [ ]:
df_single_val_repairs[
     #(df_single_val_repairs['no_separator'] == False)  
    # (df_single_val_repairs['T-box separator added to constraint'] == False)
    (df_single_val_repairs['Constraint Deleted'] == False)
    & (df_single_val_repairs['Constraint Deprecated'] == False)
    & (df_single_val_repairs['Included as Exception'] == False)
    & (df_single_val_repairs['A-box statement Deleted'] == False) 
    & (df_single_val_repairs['A-box complementary statement Deleted'] == False) #
    & (df_single_val_repairs['A-box separator added'] == False) 
    & (df_single_val_repairs['T-box separator added'] == False) 
     
    ]

In [ ]:
df_single_val_repairs.iloc[240224]

In [ ]:
df_single_val_repairs.iloc[240224]['wds1']

In [ ]:
df_single_val_repairs['A-box complementary statement separator added'] = None

In [ ]:
def checkIfComplementaryHadSeparadorAdded(row):
    
    # URL of the endpoint
    endpoint_19 = "ENTER_qEndpoint_WD_2019"
    endpoint_23 = "ENTER_qEndpoint_WD_2023"
    
    separators = getConstraintSeparator(row['property'])
    stmts = getStatementsList(row)
        
    if separators and stmts:
        for stmt in stmts:
            for separator in separators:
                if not hasSeparator(stmt, separator, endpoint_19) and hasSeparator(stmt, separator, endpoint_23):
                      return True
    return False
    
checkIfComplementaryHadSeparadorAdded(df_single_val_repairs.iloc[240224])

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os


# Process the rows with progress bar
for index, row in tqdm(df_single_val_repairs.iterrows(), total=len(df_single_val_repairs)):
    
    # Check for unprocessed rows
    if row["A-box statement Deleted"] is False:
        if row['no_separator'] is False:      
            result = checkIfComplementaryHadSeparadorAdded(row)
            df_single_val_repairs.at[index, 'A-box complementary statement separator added'] = result
        else:
            df_single_val_repairs.at[index, 'A-box complementary statement separator added'] = False
        
        # Save a checkpoint every 10,000 rows
        if index % 100000 == 0:
            df_single_val_repairs.to_csv(checkpoint_file, index=False)
            print(f"Checkpoint saved at row {index}.")
    else:
        df_single_val_repairs.at[index, 'A-box complementary statement separator added'] = False
        
            
# Save the final output
df_single_val_repairs.to_csv(checkpoint_file, index=False)
print("Processing complete. Final output saved.")

In [ ]:
df_single_val_repairs["A-box complementary statement separator added"].value_counts()

In [ ]:
df_single_val_repairs[
     #(df_single_val_repairs['no_separator'] == False)  
    # (df_single_val_repairs['T-box separator added to constraint'] == False)
    (df_single_val_repairs['Constraint Deleted'] == False)
    & (df_single_val_repairs['Constraint Deprecated'] == False)
    & (df_single_val_repairs['Included as Exception'] == False)
    & (df_single_val_repairs['A-box statement Deleted'] == False) 
    & (df_single_val_repairs['A-box complementary statement Deleted'] == False) #
    & (df_single_val_repairs['A-box separator added'] == False) 
    & (df_single_val_repairs['T-box separator added'] == False) 
    & (df_single_val_repairs['A-box complementary statement separator added'] == False) 
     
    ]

In [ ]:
df_single_val_repairs.to_csv("single_value_all_repairs.csv",index=False)

In [ ]:
# Select the two columns you want to print
print(df_single_val_repairs[['T-box separator added to constraint', 'T-box separator added']])


In [ ]:
df_single_val_repairs.iloc[0]['wds1']

In [ ]:
df_single_val_repairs.iloc[0]

In [ ]:
testTboxseparatorAdded(df_single_val_repairs.iloc[0])

In [ ]:
df_single_val_repairs = df_single_val_repairs.rename(columns={'T-box separator added': 'T-box separator added of existing A-box separator'})

In [ ]:
df_single_val_repairs = df_single_val_repairs.rename(columns={'T-box separator added of existing A-box separator': 'T-box separator added of existing A-box qualifier'})

In [ ]:
df_single_val_repairs.iloc[675336]

In [ ]:
df_single_val_repairs.iloc[675336]['wds1']

In [ ]:
df_single_val_repairs = df_single_val_repairs.rename(columns={'T-box separator added to constraint': 'T-box separator added to constraint with 0 previous separators'})

In [ ]:
df_single_val_repairs.iloc[675336]

In [ ]:
df_single_val_repairs['T-box separator added'] = df_single_val_repairs['T-box separator added to constraint with 0 previous separators'] | df_single_val_repairs['T-box separator added of existing A-box qualifier']


In [ ]:
df_single_val_repairs['T-box separator added'].value_counts()

In [ ]:
df_single_val_repairs

In [ ]:
df_single_val_repairs_P21 =  df_single_val_repairs[(df_single_val_repairs['property'] == 'http://www.wikidata.org/entity/P21')]

In [ ]:
df_single_val_repairs_P21.to_csv("single_value_P21_repairs.csv",index=False)

In [ ]:
df_single_val_repairs_P21